# Fase 3 — Programación Orientada a Objetos

**Asignatura:** MCDI500 — Introducción a la Programación  
**Proyecto:** Detección de Anomalías y Posibles Fraudes en Permisos de Circulación Vehicular  
**Stack:** Python 3.14 · pandas · scikit-learn · loguru

# Proyecto de Magíster: Detección de Anomalías y Posibles Fraudes en Permisos de Circulación

## Problemática
En la administración municipal, el cobro de los Permisos de Circulación Vehicular depende fuertemente de datos ingresados manualmente y tasaciones. Debido a la falta de validación estandarizada en los sistemas locales, existen múltiples errores de digitación, inconsistencias de formato y, potencialmente, **fraudes y evasiones** (por ejemplo, vehículos de alto valor comercial registrados con características alteradas para pagar menos, o modificaciones vehiculares no declaradas). Esta desestructuración de la información genera ineficiencias en la gestión pública y grandes pérdidas de ingresos municipales.

## Objetivo General
Diseñar e implementar soluciones algorítmicas en Python aplicadas al proyecto transversal, integrando principios de programación estructurada, recursiva y orientada a objetos para construir código modular, reutilizable y escalable.

## Configuración del Entorno

Importación de librerías externas e internas. `loguru` reemplaza el módulo `logging` estándar entregando timestamps, niveles de color y formato estructurado sin configuración adicional.

In [1]:
import os
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from loguru import logger
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler

## Arquitectura de Clases


Cada clase tiene **una sola responsabilidad** (alta cohesión) y expone un único método público orquestador que ejecuta los pasos privados en secuencia (bajo acoplamiento entre módulos).

### Clase `Explorador`

Analiza el dataset en crudo: dimensiones, tipos de datos, nulos, duplicados, tipos mixtos, anomalías de texto y coherencia de fechas. Todos los análisis son métodos privados (`_`); `explorar()` los ejecuta en orden y es el único punto de entrada público.

In [2]:
class Explorador:
    """Clase para realizar un análisis exploratorio inicial del dataset,
    enfocándose en la estructura, calidad y posibles anomalías.
    Cada método privado se encarga de un aspecto específico del análisis,
    y el método público 'explorar' los ejecuta en secuencia."""

    def __init__(self, df):
        self.df = df

    def _find_col(self, *names):
        """Retorna el primer nombre de columna que exista en el df,
        usando comparación case-insensitive. Funciona antes y después de codificar()."""
        lower_map = {c.lower(): c for c in self.df.columns}
        for name in names:
            found = lower_map.get(name.lower())
            if found:
                return found
        return None

    def _explorar_estructura(self):
        logger.info("Estructura del dataset")
        print(f"Dimensiones: {self.df.shape[0]} filas x {self.df.shape[1]} columnas\n")
        display(self.df.head(10))
        print("\n--- Tipos de datos ---")
        display(self.df.dtypes.rename('tipo').to_frame())
        print("\n--- Estadísticas descriptivas ---")
        display(self.df.describe())

    def _analizar_categorias(self, columnas=None):
        logger.info("Frecuencias por columna categórica")
        if columnas is None:
            columnas = self.df.select_dtypes(include='object').columns.tolist()
        for col in columnas:
            print(f"\n{col}:")
            print(self.df[col].value_counts(dropna=False))
            print("-" * 30)

    def _generar_reporte_nulos(self):
        logger.info("Reporte de valores nulos")
        reporte = []
        for col in self.df.columns:
            nulos = self.df[col].isnull().sum()
            reporte.append({
                'Columna': col,
                'Tipo': str(self.df[col].dtype),
                'Nulos': nulos,
                '% Nulos': f"{nulos / len(self.df) * 100:.2f}%"
            })
        df_reporte = pd.DataFrame(reporte)
        con_nulos = df_reporte[df_reporte['Nulos'] > 0].sort_values('Nulos', ascending=False)
        display(con_nulos)
        return df_reporte

    def _detectar_duplicados(self):
        n = self.df.duplicated().sum()
        if n > 0:
            logger.warning(f"Filas totalmente duplicadas: {n}")
            col_id = self._find_col('_id', 'id')
            df_dup = self.df[self.df.duplicated(keep=False)]
            display(df_dup.sort_values(col_id).head(10) if col_id else df_dup.head(10))
        else:
            logger.success("Sin filas duplicadas")
        return n

    def _detectar_tipos_mixtos(self, columnas_numericas):
        logger.info("Detección de tipos mixtos en columnas numéricas")
        for col in [c for c in columnas_numericas if c in self.df.columns]:
            temp = pd.to_numeric(self.df[col], errors='coerce')
            mascara = self.df[col].notnull() & temp.isnull()
            if mascara.any():
                logger.warning(f"'{col}': {self.df[mascara][col].unique()} ({mascara.sum()} filas con valores no numéricos)")
            else:
                logger.success(f"'{col}': OK")

    def _analizar_anomalias_texto(self):
        logger.info("Longitud máxima en columnas de texto")
        cols_texto = [self._find_col(c) for c in ['TipoVehiculo', 'Marca', 'Modelo']]
        cols_texto = [c for c in cols_texto if c]
        for col in cols_texto:
            max_len = self.df[col].astype(str).map(len).max()
            if max_len > 50:
                logger.warning(f"'{col}': {max_len} caracteres (supera umbral)")
                display(self.df[self.df[col].astype(str).map(len) > 50][[col]].head())
            else:
                logger.success(f"'{col}': {max_len} caracteres")
        col_tipo = self._find_col('TipoVehiculo')
        col_cil  = self._find_col('Cilindrada')
        if col_tipo and col_cil:
            logger.info(f"Cilindrada promedio por {col_tipo}")
            resumen = self.df.groupby(col_tipo)[col_cil].agg(['mean', 'max', 'min'])
            display(resumen.sort_values('mean', ascending=False))

    def _validar_coherencia_fechas(self):
        col_fe    = self._find_col('Fecha_Emision')
        col_fv    = self._find_col('Fecha_Vencimiento')
        col_id    = self._find_col('_id', 'id')
        col_anio  = self._find_col('Ano_Fabricacion')
        col_marca = self._find_col('Marca')
        col_mod   = self._find_col('Modelo')

        if not col_fe or not col_fv:
            logger.warning("Columnas de fecha no encontradas — validación omitida")
            return 0

        fe = pd.to_datetime(self.df[col_fe], errors='coerce')
        fv = pd.to_datetime(self.df[col_fv], errors='coerce')
        mascara = fe > fv
        n = mascara.sum()
        if n > 0:
            logger.warning(f"Registros con {col_fe} > {col_fv}: {n}")
            cols_disp = [c for c in [col_id, col_fe, col_fv, col_anio, col_marca, col_mod] if c]
            display(self.df[mascara][cols_disp].head(10))
            if col_anio:
                plt.figure(figsize=(10, 5))
                sns.histplot(self.df[mascara][col_anio], bins=15, kde=True, color='orange')
                plt.title('Año de Fabricación — Registros con Fechas Inconsistentes')
                plt.xlabel('Año de Fabricación')
                plt.ylabel('Frecuencia')
                plt.grid(axis='y', linestyle='--', alpha=0.7)
                plt.tight_layout()
                plt.show()
        else:
            logger.success("Coherencia de fechas: OK")
        return n

    def explorar(self, columnas_numericas=None):
        if columnas_numericas is None:
            columnas_numericas = self.df.select_dtypes(include='number').columns.tolist()
        self._explorar_estructura()
        self._analizar_categorias()
        self._generar_reporte_nulos()
        self._detectar_duplicados()
        self._detectar_tipos_mixtos(columnas_numericas)
        self._analizar_anomalias_texto()
        self._validar_coherencia_fechas()

### Jerarquía de Transformadores: Herencia y Polimorfismo


`Transformador` es una clase abstracta (`ABC`) que define el contrato: toda subclase debe implementar `aplicar(df) → DataFrame`. Esto garantiza una interfaz uniforme sin importar la estrategia concreta.

```
Transformador  (ABC — contrato)
├── ImputarModa          → rellena con la moda de la columna
├── ImputarMediana       → rellena con mediana, opcionalmente agrupando por categoría
├── ImputarValorFijo     → rellena con un valor literal definido por el usuario
├── EscalarZScore        → StandardScaler  (media=0, std=1)
└── EscalarMinMax        → MinMaxScaler    (rango [0, 1])
```

El `Limpiador` y el `Preprocesador` iteran `for t in transformadores: df = t.aplicar(df)` sin conocer el tipo concreto — ese es el **polimorfismo**.

In [3]:
from abc import ABC, abstractmethod
from sklearn.preprocessing import StandardScaler, MinMaxScaler


class Transformador(ABC):
    """Contrato común para todos los transformadores del pipeline.

    Cualquier clase que herede debe implementar aplicar(self, df)
    y retornar el DataFrame modificado.
    """

    @abstractmethod
    def aplicar(self, df):
        """Aplica la transformación sobre df y lo retorna."""
        ...


# ── Imputación ───────────────────────────────────────────────────────────────

class ImputarModa(Transformador):
    """Imputa nulos de columnas categóricas con la moda."""

    def __init__(self, columnas):
        self.columnas = columnas if isinstance(columnas, list) else [columnas]

    def aplicar(self, df):
        for col in self.columnas:
            if col in df.columns:
                df[col] = df[col].fillna(df[col].mode()[0])
                logger.info(f"ImputarModa: '{col}' imputada con moda")
        return df


class ImputarMediana(Transformador):
    """Imputa nulos numéricos con la mediana, opcionalmente agrupando por otra columna."""

    def __init__(self, columna, grupo=None):
        self.columna = columna
        self.grupo   = grupo

    def aplicar(self, df):
        if self.grupo and self.grupo in df.columns:
            df[self.columna] = (
                df.groupby(self.grupo)[self.columna]
                .transform(lambda x: x.fillna(x.median()))
            )
        df[self.columna] = df[self.columna].fillna(df[self.columna].median())
        logger.info(f"ImputarMediana: '{self.columna}' imputada con mediana")
        return df


class ImputarValorFijo(Transformador):
    """Imputa nulos con un valor fijo por columna."""

    def __init__(self, columnas_valores: dict):
        self.columnas_valores = columnas_valores

    def aplicar(self, df):
        for col, valor in self.columnas_valores.items():
            if col in df.columns:
                df[col] = df[col].fillna(valor)
                logger.info(f"ImputarValorFijo: '{col}' → '{valor}'")
        return df



class ConsolidarCategorias(Transformador):
    """Reemplaza alias en una columna categórica mediante un mapeo."""

    def __init__(self, columna: str, mapeo: dict):
        self.columna = columna
        self.mapeo   = mapeo

    def aplicar(self, df):
        if self.columna in df.columns:
            df[self.columna] = df[self.columna].replace(self.mapeo)
            logger.info(f"ConsolidarCategorias: '{self.columna}' consolidada")
        return df

# ── Escalado ─────────────────────────────────────────────────────────────────

class EscalarZScore(Transformador):
    """Escala columnas numéricas con z-score (media=0, std=1)."""

    def __init__(self, columnas):
        self.columnas = columnas
        self.scaler   = StandardScaler()

    def aplicar(self, df):
        cols = [c for c in self.columnas if c in df.columns]
        df[cols] = self.scaler.fit_transform(df[cols])
        logger.info(f"EscalarZScore aplicado a: {cols}")
        return df


class EscalarMinMax(Transformador):
    """Escala columnas numéricas al rango [0, 1]."""

    def __init__(self, columnas):
        self.columnas = columnas
        self.scaler   = MinMaxScaler()

    def aplicar(self, df):
        cols = [c for c in self.columnas if c in df.columns]
        df[cols] = self.scaler.fit_transform(df[cols])
        logger.info(f"EscalarMinMax aplicado a: {cols}")
        return df

### Clase `Limpiador`

Responsabilidad única: **limpieza estructural**. Elimina columnas sin aporte analítico (`Equipamiento` con >55% vacíos, `Tonelaje` con >98% ceros) y convierte columnas numéricas al tipo correcto, generando `NaN` donde el valor no es numérico.

La normalización de texto/fechas y la consolidación de categorías se delegan al pipeline via `codificar()` y `consolidar()`.

In [4]:
class Limpiador:
    """Limpieza estructural del DataFrame: elimina columnas irrelevantes
    y convierte tipos de datos. La normalización de texto/fechas y la
    consolidación de categorías se delegan al pipeline.
    """

    def __init__(self, df):
        self.df = df.copy()

    def _eliminar_columna_equipamiento(self):
        # >55% sin información útil (nulos + 'EQUI' sin definición)
        self.df = self.df.drop(columns=['Equipamiento'], errors='ignore')
        logger.info("Columna 'Equipamiento' eliminada")

    def _eliminar_columna_tonelaje(self):
        # >98% de los registros tienen valor 0, sin aporte analítico
        self.df = self.df.drop(columns=['Tonelaje'], errors='ignore')
        logger.info("Columna 'Tonelaje' eliminada")

    def _convertir_numericas(self):
        """Los valores no convertibles pasan a NaN para ser imputados después."""
        columnas = ['Valor_Contado', 'Total_a_Pagar', 'Ano_Fabricacion', 'Cilindrada', 'Tonelaje']
        for col in [c for c in columnas if c in self.df.columns]:
            antes = self.df[col].isna().sum()
            self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
            convertidos = self.df[col].isna().sum() - antes
            if convertidos > 0:
                logger.warning(f"'{col}': {convertidos} valor(es) no numérico(s) convertidos a NaN")

    def limpiar(self):
        self._eliminar_columna_equipamiento()
        self._eliminar_columna_tonelaje()
        self._convertir_numericas()
        logger.success(f"Limpieza estructural — {self.df.shape[0]} filas x {self.df.shape[1]} columnas")
        return self.df


### Suite de Pruebas


Prueba exhaustiva del método `validar()` del `Preprocesador` cubriendo tres categorías:

| Tipo | Qué verifica |
|---|---|
| **Normal** | Años y valores dentro de rangos esperados — debe pasar sin error |
| **Límite** | Valores en los bordes del rango válido — comportamiento en el extremo |
| **Excepción** | Entradas inválidas — debe lanzar `AssertionError` |

Cada caso usa `assert` para verificar el resultado esperado y reporta `[PASS]` / `[FAIL]` con `loguru`.

In [5]:
class TestValidaciones:
    """Suite de pruebas para validar que el método validar() de Preprocesador
    detecta correctamente casos normales, límite y de excepción."""

    @staticmethod
    def _chk(cond, msg):
        if not cond:
            raise AssertionError(msg)

    @staticmethod
    def _caso(nombre, fn):
        try:
            fn()
            logger.success(f"[PASS] {nombre}")
        except AssertionError as e:
            logger.warning(f"[FAIL] {nombre} → {e}")
        except Exception as e:
            logger.error(f"[ERROR] {nombre} → {e}")

    def ejecutar(self, anio_actual=2026):
        logger.info("── Iniciando suite de pruebas de validar ──")

        # ── Caso normal ───────────────────────────────────────────────────
        df_ok = pd.DataFrame({
            'ano_fabricacion': [2000, 2010, 2020],
            'cilindrada':      [1600, 2000, 1400],
        })
        self._caso(
            "Normal — sin nulos, sin duplicados, años y cilindradas válidos",
            lambda: (
                self._chk(df_ok.isna().sum().sum() == 0,                             "nulos inesperados"),
                self._chk(not df_ok.duplicated().any(),                              "duplicados inesperados"),
                self._chk(df_ok['ano_fabricacion'].between(1900, anio_actual).all(), "año fuera de rango"),
                self._chk((df_ok['cilindrada'] >= 0).all(),                         "cilindrada negativa"),
            )
        )

        # ── Casos límite ──────────────────────────────────────────────────
        df_limite = pd.DataFrame({
            'ano_fabricacion': [1900, anio_actual],
            'cilindrada':      [0, 9999],
        })
        self._caso(
            f"Límite — ano_fabricacion=[1900, {anio_actual}], cilindrada=[0, 9999]",
            lambda: (
                self._chk(df_limite['ano_fabricacion'].between(1900, anio_actual).all(), "año fuera de rango"),
                self._chk((df_limite['cilindrada'] >= 0).all(),                          "cilindrada negativa"),
            )
        )

        # ── Casos de excepción ────────────────────────────────────────────
        df_nulos = pd.DataFrame({'ano_fabricacion': [2000, None], 'cilindrada': [1600, 1400]})
        self._caso(
            "Excepción — nulos presentes (debe fallar)",
            lambda: self._chk(df_nulos.isna().sum().sum() == 0, "Existen valores nulos")
        )

        df_duplicados = pd.DataFrame({'ano_fabricacion': [2000, 2000], 'cilindrada': [1600, 1600]})
        self._caso(
            "Excepción — filas duplicadas (debe fallar)",
            lambda: self._chk(not df_duplicados.duplicated().any(), "Existen filas duplicadas")
        )

        df_anio_fuera = pd.DataFrame({'ano_fabricacion': [1800, 2050], 'cilindrada': [1600, 1400]})
        self._caso(
            f"Excepción — años fuera de rango [1800, 2050] (debe fallar)",
            lambda: self._chk(df_anio_fuera['ano_fabricacion'].between(1900, anio_actual).all(),
                              "Años de fabricación fuera de rango")
        )

        df_cilindrada_neg = pd.DataFrame({'ano_fabricacion': [2000, 2010], 'cilindrada': [1600, -100]})
        self._caso(
            "Excepción — cilindrada negativa (debe fallar)",
            lambda: self._chk((df_cilindrada_neg['cilindrada'] >= 0).all(), "Cilindradas negativas detectadas")
        )

        logger.info("── Suite de pruebas finalizada ──")

### Clase `Preprocesador` — Orquestador del Pipeline

Coordina todas las clases anteriores en un **pipeline fluido** mediante encadenamiento de métodos (*method chaining*): cada método retorna `self`, permitiendo llamadas consecutivas sobre el mismo objeto sin variables intermedias.

```python
Preprocesador(ruta)
    .cargar_datos()
    .limpiar()          # elimina columnas, convierte tipos
    .imputar()          # ImputarModa / Mediana / ValorFijo
    .consolidar()       # ConsolidarCategorias
    .codificar()        # normaliza texto, fechas y snake_case
    .escalar([...])     # usa EscalarZScore / EscalarMinMax
    .validar()          # misma lógica que TestValidaciones
    .exportar(ruta)
    .resultado()        # retorna el DataFrame final
```

In [6]:
class Preprocesador:
    """Objeto que guarda una tabla de datos y sabe prepararla para el analisis.

    Cada metodo del pipeline retorna self para permitir encadenamiento fluido:
        df = Preprocesador(ruta).cargar_datos().limpiar().imputar().consolidar().codificar().resultado()
    """

    def __init__(self, ruta):
        self.ruta   = ruta
        self.df     = None
        self.scaler = None

    def cargar_datos(self):
        """Carga el dataset desde disco y lo guarda en self.df."""
        try:
            if not os.path.exists(self.ruta):
                raise FileNotFoundError(f"No se encontró el archivo: '{self.ruta}'")
            self.df = pd.read_csv(self.ruta, encoding='utf-8')
            logger.success(f"Dataset cargado — {self.df.shape[0]} filas x {self.df.shape[1]} columnas")
            return self
        except Exception as e:
            logger.error(f"cargar_datos falló: {e}")
            return self

    def explorar(self, columnas_numericas=None):
        """Análisis exploratorio inicial del dataset."""
        try:
            Explorador(self.df).explorar(columnas_numericas)
        except Exception as e:
            logger.error(f"explorar falló: {e}")
        return self

    def limpiar(self):
        """Limpieza y transformación de los datos."""
        try:
            self.df = Limpiador(self.df).limpiar()
        except Exception as e:
            logger.error(f"limpiar falló: {e}")
        return self

    def imputar(self):
        """Imputa nulos via Transformadores — polimorfismo en acción."""
        try:
            for t in [
                ImputarModa(['Combustible', 'Transmision']),
                ImputarMediana('Cilindrada', grupo='TipoVehiculo'),
                ImputarValorFijo({'Codigo_SII': 'SIN_CODIGO', 'Comuna_Anterior': 'OTRA'}),
            ]:
                self.df = t.aplicar(self.df)
            self.df['Cilindrada'] = self.df['Cilindrada'].astype(int)
            logger.success("Imputación completada")
        except Exception as e:
            logger.error(f"imputar falló: {e}")
        return self

    def consolidar(self):
        """Consolida alias en columnas categóricas via ConsolidarCategorias."""
        try:
            for t in [
                ConsolidarCategorias('TipoVehiculo', {
                    'SEDAN': 'AUTOMOVIL',      'SEDAN 2': 'AUTOMOVIL',
                    'SUV': 'STATION WAGON',    'SUV 2': 'STATION WAGON',
                    'MOTO1': 'MOTO',           'MOTO2': 'MOTO',
                    'VAN 2': 'FURGON',
                    'MINIBUS PARTICULAR':         'MINIBUS',
                    'MINIBUS TRANS  PASAJERO':    'MINIBUS',
                    'MINIBUS TRANS PASAJERO':     'MINIBUS',
                    'MINIBUS DE TURISMO':         'MINIBUS',
                    'MINIBUS ESCOLAR':            'MINIBUS',
                }),
                ConsolidarCategorias('Transmision', {'CVT': 'AUT'}),
                ConsolidarCategorias('Combustible', {'MEC': 'DIES'}),
            ]:
                self.df = t.aplicar(self.df)
            logger.success("Categorías consolidadas")
        except Exception as e:
            logger.error(f"consolidar falló: {e}")
        return self

    def codificar(self):
        """Normaliza texto y fechas, luego convierte columnas a snake_case."""
        try:
            def _sin_tildes(texto):
                if not isinstance(texto, str):
                    return texto
                return "".join(
                    c for c in unicodedata.normalize('NFD', texto)
                    if not unicodedata.combining(c)
                )

            cols_texto = [
                'TipoVehiculo', 'Marca', 'Modelo', 'Combustible', 'Transmision',
                'Metodo de Pago', 'Cuotas Permiso', 'Comuna_Propietario',
                'Comuna_Permiso', 'Comuna_Anterior'
            ]
            for col in [c for c in cols_texto if c in self.df.columns]:
                self.df[col] = self.df[col].str.strip().str.upper().apply(_sin_tildes)
            logger.info("Texto normalizado")

            for col in ['Fecha_Emision', 'Fecha_Vencimiento']:
                if col in self.df.columns:
                    self.df[col] = pd.to_datetime(self.df[col], errors='coerce').dt.date
            logger.info("Fechas normalizadas")

            def _a_snake_case(nombre):
                s = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', nombre)
                s = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s).lower()
                return s.replace(' ', '_').replace('.', '').replace('__', '_').strip('_')
            self.df.columns = [_a_snake_case(col) for col in self.df.columns]
            logger.success(f"Codificación completada — columnas: {self.df.columns.tolist()}")
        except Exception as e:
            logger.error(f"codificar falló: {e}")
        return self
    
    def escalar(self, columnas, metodo='zscore'):
        """Escala columnas numéricas usando un Transformador según el método elegido.

        metodo='zscore'  → EscalarZScore  (media=0, std=1)
        metodo='minmax'  → EscalarMinMax  (rango [0, 1])
        """
        try:
            transformadores = {
                'zscore': EscalarZScore(columnas),
                'minmax': EscalarMinMax(columnas),
            }
            if metodo not in transformadores:
                raise ValueError(f"Método desconocido: '{metodo}'. Usa 'zscore' o 'minmax'.")
            self.df = transformadores[metodo].aplicar(self.df)
        except Exception as e:
            logger.error(f"escalar falló: {e}")
        return self

    def validar(self, anio_actual=2026):
        """Verifica condiciones mínimas de calidad del dataset antes de exportar."""
        TestValidaciones().ejecutar(anio_actual)

        logger.info("── Validando dataset real ──")
        try:
            assert self.df.isna().sum().sum() == 0,                             "Existen valores nulos"
            assert not self.df.duplicated().any(),                              "Existen filas duplicadas"
            assert self.df['ano_fabricacion'].between(1900, anio_actual).all(), "Años de fabricación fuera de rango"
            assert (self.df['cilindrada'] >= 0).all(),                          "Cilindradas negativas detectadas"
            logger.success("Dataset real: todas las validaciones pasaron")
        except AssertionError as e:
            logger.error(f"validar falló: {e}")
        except Exception as e:
            logger.error(f"validar — error inesperado: {e}")
        return self

    def exportar(self, ruta):
        """Guarda el dataset procesado en disco."""
        try:
            os.makedirs(os.path.dirname(ruta), exist_ok=True)
            self.df.to_csv(ruta, index=False, encoding='utf-8')
            kb = os.path.getsize(ruta) / 1024
            logger.success(f"Dataset exportado → {ruta}  ({kb:.1f} KB | {self.df.shape[0]} filas x {self.df.shape[1]} columnas)")
        except Exception as e:
            logger.error(f"exportar falló: {e}")
        return self
    


    def resultado(self):
        """Retorna el DataFrame procesado al final del pipeline."""
        return self.df

### Documentación de Arquitectura


La arquitectura sigue el principio de **una responsabilidad por componente**. La tabla resume cada pieza y el patrón de diseño que la guía:

| Componente | Responsabilidad | Patrón aplicado |
|---|---|---|
| `Explorador` | Análisis exploratorio del dataset | Encapsulamiento de estado |
| `Limpiador` | Limpieza, casting e imputación | Facade (delega a `Transformadores`) |
| `Transformador` (ABC) | Contrato común de transformación | Template Method |
| `ImputarModa / Mediana / ValorFijo` | Estrategias de imputación intercambiables | Strategy |
| `EscalarZScore / MinMax` | Estrategias de escalado intercambiables | Strategy |
| `TestValidaciones` | Verificación con casos normal / límite / excepción | Test Suite |
| `Preprocesador` | Orquestador del pipeline completo | Fluent Interface (*method chaining*) |

**Decisiones técnicas y su justificación:**

- **`_` en atributos y métodos**: protege el estado interno; solo los métodos públicos son la interfaz — *encapsulamiento*
- **`@abstractmethod` en `Transformador`**: garantiza que toda subclase implemente `aplicar(df)` — el error de diseño se detecta al instanciar, no en tiempo de ejecución
- **`return self`**: habilita el *method chaining* sin variables intermedias; hace el pipeline legible como secuencia de pasos
- **`loguru`** en lugar de `print` o `logging`: contexto automático (timestamp, nivel, módulo) sin boilerplate en cada clase; `print` se reserva para salidas visuales/tabulares

## Preprocesamiento y Ejecución del Pipeline


El pipeline ejecuta el preprocesamiento completo en cuatro pasos: carga y limpieza del dataset, escalado de variables numéricas, validación de coherencia y exportación del resultado. Cada paso corre en celda separada para inspeccionar los logs de forma independiente.

La variable `pipe` mantiene el estado entre celdas gracias a que cada método retorna `self` (*Fluent Interface*). Todas las decisiones de transformación quedan trazables en los logs de `loguru`.

### Paso 1 — Carga, Limpieza e Imputación

Ejecuta la cadena completa de preparación en orden:

1. **`limpiar()`** — elimina columnas sin aporte analítico y convierte tipos de datos
2. **`imputar()`** — rellena nulos con moda, mediana o valor fijo vía `Transformadores`
3. **`consolidar()`** — unifica aliases categóricos (`SEDAN→AUTOMOVIL`, `CVT→AUT`, ...) vía `ConsolidarCategorias`
4. **`codificar()`** — normaliza texto (mayúsculas, sin tildes), fechas y renombra columnas a `snake_case`

In [7]:
# Paso 1: limpiar → imputar → consolidar → codificar
pipe = (Preprocesador("../datos/original/permiso-circulacion-2026.csv")
        .cargar_datos()
        .limpiar()
        .imputar()
        .consolidar()
        .codificar())

2026-06-11 00:20:39.069 | SUCCESS  | __main__:cargar_datos:19 - Dataset cargado — 3195 filas x 21 columnas
2026-06-11 00:20:39.070 | INFO     | __main__:_eliminar_columna_equipamiento:13 - Columna 'Equipamiento' eliminada
2026-06-11 00:20:39.071 | INFO     | __main__:_eliminar_columna_tonelaje:18 - Columna 'Tonelaje' eliminada
2026-06-11 00:20:39.071 | SUCCESS  | __main__:limpiar:34 - Limpieza estructural — 3195 filas x 19 columnas
2026-06-11 00:20:39.072 | INFO     | __main__:aplicar:30 - ImputarModa: 'Combustible' imputada con moda
2026-06-11 00:20:39.072 | INFO     | __main__:aplicar:30 - ImputarModa: 'Transmision' imputada con moda
/Users/alonso/Documents/GitHub/unab-magister-pro-ciencia-datos-MCDI500/.venv/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/alonso/Documents/GitHub/unab-magister-pro-ciencia-datos-MCDI500/.venv/lib/python3.14/site-packages/numpy/lib/_na

### Paso 2 — Escalado de Variables Numéricas

Estandariza variables numéricas con dos estrategias vía polimorfismo:
- **z-score** (`EscalarZScore`): media=0, std=1 — para modelos que asumen distribución normal
- **min-max** (`EscalarMinMax`): rango [0,1] — para variables acotadas o redes neuronales

In [8]:
# Paso 2: escalar variables numéricas
pipe = (pipe
        .escalar(['valor_contado', 'cilindrada', 'ano_fabricacion'])   # z-score
        .escalar(['total_a_pagar'], metodo='minmax'))                  # [0, 1]

2026-06-11 00:20:39.105 | INFO     | __main__:aplicar:92 - EscalarZScore aplicado a: ['valor_contado', 'cilindrada', 'ano_fabricacion']
2026-06-11 00:20:39.106 | INFO     | __main__:aplicar:106 - EscalarMinMax aplicado a: ['total_a_pagar']


### Paso 3 — Validación

Verifica coherencia interna del dataset: rangos de años, valores positivos y consistencia de fechas. Comparte la lógica con la suite `TestValidaciones` — si pasa aquí, pasa allá.

In [9]:
# Paso 3: validar coherencia de los datos
pipe = pipe.validar()

2026-06-11 00:20:39.109 | INFO     | __main__:ejecutar:21 - ── Iniciando suite de pruebas de validar ──
2026-06-11 00:20:39.110 | SUCCESS  | __main__:_caso:14 - [PASS] Normal — sin nulos, sin duplicados, años y cilindradas válidos
2026-06-11 00:20:39.111 | SUCCESS  | __main__:_caso:14 - [PASS] Límite — ano_fabricacion=[1900, 2026], cilindrada=[0, 9999]
2026-06-11 00:20:39.111 | WARNING  | __main__:_caso:16 - [FAIL] Excepción — nulos presentes (debe fallar) → Existen valores nulos
2026-06-11 00:20:39.112 | WARNING  | __main__:_caso:16 - [FAIL] Excepción — filas duplicadas (debe fallar) → Existen filas duplicadas
2026-06-11 00:20:39.112 | WARNING  | __main__:_caso:16 - [FAIL] Excepción — años fuera de rango [1800, 2050] (debe fallar) → Años de fabricación fuera de rango
2026-06-11 00:20:39.112 | WARNING  | __main__:_caso:16 - [FAIL] Excepción — cilindrada negativa (debe fallar) → Cilindradas negativas detectadas
2026-06-11 00:20:39.112 | INFO     | __main__:ejecutar:77 - ── Suite de prue

### Paso 4 — Exportar y Resultado Final

Persiste el DataFrame procesado en `datos/resultado/df_clean.csv` y retorna el objeto `DataFrame` para continuar el análisis en celdas siguientes.

In [10]:
# Paso 4: exportar CSV y extraer DataFrame final
df_listo = (pipe
            .exportar("../datos/resultado/df_clean.csv")
            .resultado())
df_listo.head()

2026-06-11 00:20:39.130 | SUCCESS  | __main__:exportar:156 - Dataset exportado → ../datos/resultado/df_clean.csv  (665.7 KB | 3195 filas x 19 columnas)


,id,fecha_emision,ano_proceso,metodo_de_pago,cuotas_permiso,codigo_sii,comuna_propietario,comuna_permiso,valor_contado,total_a_pagar,ano_fabricacion,tipo_vehiculo,marca,modelo,cilindrada,combustible,transmision,comuna_anterior,fecha_vencimiento
0,1,2026-03-20,2026,PRESENCIAL,TOTAL,SU188001078,AISEN,RIO IBANEZ,-0.597386,0.021412,-4.345129,AUTOMOVIL,PLYMOUTH,VOLARE,1.885109,BENC,MEC,RIO IBANEZ,2026-03-31
1,2,2026-03-20,2026,PRESENCIAL,TOTAL,SU188001078,AISEN,RIO IBANEZ,-0.589455,0.019972,-4.345129,AUTOMOVIL,PLYMOUTH,VOLARE,1.885109,BENC,MEC,RIO IBANEZ,2027-03-31
2,3,2026-03-20,2026,PRESENCIAL,TOTAL,SU188001078,AISEN,RIO IBANEZ,-0.616718,0.024668,-4.345129,AUTOMOVIL,PLYMOUTH,VOLARE,1.885109,BENC,MEC,RIO IBANEZ,2024-03-31
3,4,2026-03-20,2026,PRESENCIAL,TOTAL,SU188001078,AISEN,RIO IBANEZ,-0.606826,0.023265,-4.345129,AUTOMOVIL,PLYMOUTH,VOLARE,1.885109,BENC,MEC,RIO IBANEZ,2025-03-31
4,5,2026-04-08,2026,PRESENCIAL,TOTAL,SIN_CODIGO,COYHAIQUE,RIO IBANEZ,0.113933,0.083078,-3.219311,CAMION,MERCEDES BENZ,L1114 48,1.785088,DIES,MEC,RIO BUENO,2026-09-30
